# Affected Population Estimation

## Objective

This notebook estimates the affected population for every disaster event after the
Impact Score prediction stage.

The estimated affected population is then divided into:

- Children
- Adults
- Elderly

The notebook also computes a Vulnerability Score that will later be used by the
resource allocation engine.

---

### Input

datasets/final_disaster_dataset_engineered.csv

---

### Output

datasets/affected_population.csv

---

### Pipeline

Final Disaster Dataset

↓

Estimate Affected Population

↓

Population Segmentation

↓

Vulnerability Score

↓

affected_population.csv

In [54]:
from pathlib import Path

import numpy as np
import pandas as pd

In [55]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_DIR = Path.cwd().parents[1]

DATA_DIR = PROJECT_DIR / "datasets"

INPUT_FILE = DATA_DIR / "final_disaster_dataset_engineered.csv"

OUTPUT_FILE = DATA_DIR / "affected_population.csv"

In [56]:
assert INPUT_FILE.exists(), "Engineered dataset not found."

print("Input Dataset Found")

print(INPUT_FILE)

Input Dataset Found
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\final_disaster_dataset_engineered.csv


In [57]:
events = pd.read_csv(INPUT_FILE)

print(events.shape)

events.head()

(2374, 25)


,SourceEventID,DisasterType,Latitude,Longitude,Magnitude,Depth,Time,Temperature,Humidity,Rainfall,...,NearestHospitalDistanceKm,HospitalCount25Km,NearestClinicDistanceKm,ClinicCount25Km,NearestSchoolDistanceKm,SchoolCount25Km,NearestFireStationDistanceKm,FireStationCount25Km,WeatherScore,InfrastructureScore
0,18,Earthquake,27.5051,92.1524,4.3,10.000,2026-06-21 00:53:20.927000,0.526455,0.831579,0.0,...,0.940875,0.000000,0.854388,0.000000,0.966349,0.000849,0.646845,0.000000,0.452678,0.426163
1,19,Earthquake,27.4598,92.0424,4.3,10.000,2026-06-21 00:52:54.610000,0.584656,0.915789,0.0,...,0.955199,0.000000,0.863531,0.000000,0.954999,0.000000,0.650855,0.000000,0.500149,0.428073
2,22,Earthquake,24.3478,94.0847,4.5,52.125,2026-06-18 15:29:24.907000,0.718254,0.915789,0.0,...,0.979486,0.005413,0.964291,0.003571,0.984801,0.007640,0.883867,0.000000,0.544681,0.478634
3,39,Earthquake,24.9682,93.0641,4.4,48.154,2026-06-11 15:40:35.593000,0.805556,0.863158,0.0,...,0.981666,0.012179,0.985625,0.005357,0.971148,0.020374,0.953640,0.025641,0.556238,0.494454
4,46,Earthquake,32.3888,76.6062,4.9,10.000,2026-06-05 16:34:36.131000,0.395503,0.978947,0.0,...,0.982899,0.004060,0.939849,0.000000,0.983533,0.007640,0.923348,0.000000,0.458150,0.480166


In [58]:
print(events.columns.tolist())

['SourceEventID', 'DisasterType', 'Latitude', 'Longitude', 'Magnitude', 'Depth', 'Time', 'Temperature', 'Humidity', 'Rainfall', 'WindSpeed', 'Population', 'PopulationDensity', 'Severity', 'ImpactScore', 'NearestHospitalDistanceKm', 'HospitalCount25Km', 'NearestClinicDistanceKm', 'ClinicCount25Km', 'NearestSchoolDistanceKm', 'SchoolCount25Km', 'NearestFireStationDistanceKm', 'FireStationCount25Km', 'WeatherScore', 'InfrastructureScore']


# Exposure Ratio Estimation

The total population surrounding a disaster location does not represent the
population directly affected by the event.

Therefore an Exposure Ratio is estimated first.

The Exposure Ratio represents the proportion of the surrounding population
expected to require disaster response.

The Exposure Ratio depends upon:

- Impact Score
- Disaster Severity
- Disaster Type
- Population Density

The estimated affected population is then calculated as

Affected Population

=

Population × Exposure Ratio

This approach provides a more realistic estimate than assuming the entire
population is affected.

In [59]:
SEVERITY_MULTIPLIER = {

    "Low": 0.60,

    "Medium": 0.80,

    "High": 1.00

}

In [60]:
DISASTER_MULTIPLIER = {

    "Earthquake": 1.00,

    "Flood": 0.90,

    "Cyclone": 0.95,

    "Landslide": 0.80,

    "Wildfire": 0.70,

    "Volcano": 0.60

}

In [61]:
events["PopulationDensityFactor"] = (

    events["PopulationDensity"]

    /

    events["PopulationDensity"].max()

)

In [62]:
events["ExposureRatio"] = (

    (
        events["ImpactScore"] / 100
    )

    *

    events["Severity"].map(SEVERITY_MULTIPLIER)

    *

    events["DisasterType"].map(DISASTER_MULTIPLIER)

    *

    (
        0.6
        +
        0.4 *
        events["PopulationDensityFactor"]
    )

)

In [63]:
events["ExposureRatio"] = (

    events["ExposureRatio"]

    .clip(

        lower=0,

        upper=1

    )

)

In [64]:
print("Population NaN :", events["Population"].isna().sum())

print("ExposureRatio NaN :", events["ExposureRatio"].isna().sum())

print("Population Inf :", np.isinf(events["Population"]).sum())

print("ExposureRatio Inf :", np.isinf(events["ExposureRatio"]).sum())

Population NaN : 0
ExposureRatio NaN : 0
Population Inf : 0
ExposureRatio Inf : 0


In [65]:
events["DisasterType"].map(DISASTER_MULTIPLIER)

0       1.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
2369    1.0
2370    1.0
2371    1.0
2372    1.0
2373    1.0
Name: DisasterType, Length: 2374, dtype: float64

In [66]:
print(events["DisasterType"].value_counts())

DisasterType
Earthquake    2374
Name: count, dtype: int64


In [67]:
print(events["Severity"].value_counts())

Severity
Low       2133
Medium     239
High         2
Name: count, dtype: int64


In [68]:
events["AffectedPopulation"] = (

    events["Population"]

    *

    events["ExposureRatio"]

).round().astype(int)

In [69]:
events["AffectedPopulation"].describe()

count    2.374000e+03
mean     8.270883e+04
std      4.937343e+05
min      0.000000e+00
25%      3.010000e+02
50%      4.392500e+03
75%      5.471325e+04
max      1.160681e+07
Name: AffectedPopulation, dtype: float64

# Population Segmentation

The affected population is divided into

- Children
- Adults
- Elderly

The percentages are approximate demographic
assumptions used only for simulation.

In [70]:
CHILD_RATIO = 0.27

ADULT_RATIO = 0.60

ELDERLY_RATIO = 0.13

In [71]:
events["Children"] = (
    events["AffectedPopulation"] *
    CHILD_RATIO
).round().astype(int)

events["Adults"] = (
    events["AffectedPopulation"] *
    ADULT_RATIO
).round().astype(int)

events["Elderly"] = (
    events["AffectedPopulation"] *
    ELDERLY_RATIO
).round().astype(int)

In [72]:
events[
    [
        "AffectedPopulation",
        "Children",
        "Adults",
        "Elderly"
    ]
].head()

,AffectedPopulation,Children,Adults,Elderly
0,3374,911,2024,439
1,5429,1466,3257,706
2,68999,18630,41399,8970
3,99816,26950,59890,12976
4,4794,1294,2876,623


# Vulnerability Score

A vulnerability score is computed using

- Children
- Elderly

because these groups generally require
greater disaster assistance.

In [73]:
events["VulnerabilityScore"] = (
    (
        events["Children"] * 1.2
        +
        events["Adults"] * 1.0
        +
        events["Elderly"] * 1.5
    )
    /
    events["AffectedPopulation"].clip(lower=1)
)

events["VulnerabilityScore"] = (
    events["VulnerabilityScore"]
    .round(2)
)

In [74]:
events[
    [
        "AffectedPopulation",
        "Children",
        "Adults",
        "Elderly",
        "VulnerabilityScore"
    ]
].head()

,AffectedPopulation,Children,Adults,Elderly,VulnerabilityScore
0,3374,911,2024,439,1.12
1,5429,1466,3257,706,1.12
2,68999,18630,41399,8970,1.12
3,99816,26950,59890,12976,1.12
4,4794,1294,2876,623,1.12


In [75]:
events.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved Successfully")

print(OUTPUT_FILE)

Saved Successfully
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\affected_population.csv


In [76]:
print(events.shape)

events.head()

(2374, 32)


,SourceEventID,DisasterType,Latitude,Longitude,Magnitude,Depth,Time,Temperature,Humidity,Rainfall,...,FireStationCount25Km,WeatherScore,InfrastructureScore,PopulationDensityFactor,ExposureRatio,AffectedPopulation,Children,Adults,Elderly,VulnerabilityScore
0,18,Earthquake,27.5051,92.1524,4.3,10.000,2026-06-21 00:53:20.927000,0.526455,0.831579,0.0,...,0.000000,0.452678,0.426163,0.000695,0.183250,3374,911,2024,439,1.12
1,19,Earthquake,27.4598,92.0424,4.3,10.000,2026-06-21 00:52:54.610000,0.584656,0.915789,0.0,...,0.000000,0.500149,0.428073,0.001100,0.186286,5429,1466,3257,706,1.12
2,22,Earthquake,24.3478,94.0847,4.5,52.125,2026-06-18 15:29:24.907000,0.718254,0.915789,0.0,...,0.000000,0.544681,0.478634,0.011665,0.223250,68999,18630,41399,8970,1.12
3,39,Earthquake,24.9682,93.0641,4.4,48.154,2026-06-11 15:40:35.593000,0.805556,0.863158,0.0,...,0.025641,0.556238,0.494454,0.017018,0.221373,99816,26950,59890,12976,1.12
4,46,Earthquake,32.3888,76.6062,4.9,10.000,2026-06-05 16:34:36.131000,0.395503,0.978947,0.0,...,0.000000,0.458150,0.480166,0.000877,0.206315,4794,1294,2876,623,1.12
